In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Defense Dataset Generator

In [4]:
!pip install ultralytics pyyaml tqdm
import os
import shutil
import random
import yaml
import torch
import cv2
import gc
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

# ==========================================
# 1. DEFENSE CONFIGURATION
# ==========================================
# We use the DOTA training set here, not the validation set.
SOURCE_IMAGES_DIR = "/kaggle/input/datasets/pawanpauljay/dota-1-5/dota/images" # Update if train images are elsewhere
SOURCE_LABELS_DIR = "/kaggle/input/datasets/pawanpauljay/dota-1-5/dota/dota_labels"

DEFENSE_DIR = "/kaggle/working/defense_dataset"
DEFENSE_IMAGES_DIR = os.path.join(DEFENSE_DIR, "images/train")
DEFENSE_LABELS_DIR = os.path.join(DEFENSE_DIR, "labels/train")

# Dataset generation parameters
NUM_DEFENSE_IMAGES = 2000  # Number of original images to sample
IMG_SIZE = 1024

# PGD Attack parameters for the defense dataset
DEFENSE_EPSILON = 0.04  # A moderate poison strength to train against
PGD_ITERS = 10
PGD_ALPHA = DEFENSE_EPSILON * 0.25

# 16 DOTA Classes
CLASS_NAMES = {
    0: 'plane', 1: 'ship', 2: 'storage tank', 3: 'baseball diamond',
    4: 'tennis court', 5: 'basketball court', 6: 'ground track field',
    7: 'harbor', 8: 'bridge', 9: 'large vehicle', 10: 'small vehicle',
    11: 'helicopter', 12: 'roundabout', 13: 'soccer ball field',
    14: 'swimming pool', 15: 'container crane'
}
NAME_TO_IDX = {v: k for k, v in CLASS_NAMES.items()}

# ==========================================
# 2. HELPER: DOTA TO YOLO OBB CONVERTER
# ==========================================
def convert_dota_to_yolo_obb(src_path, dst_path, img_width, img_height):
    """Converts raw DOTA pixel coordinates to normalized YOLO OBB format."""
    if not os.path.exists(src_path): return False
    out = []
    with open(src_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 10: continue
            try:
                coords = list(map(float, p[:8]))
                cls = p[8].replace("-", " ")
                if cls not in NAME_TO_IDX: continue
                cls_id = NAME_TO_IDX[cls]
                norm = []
                for i in range(0, 8, 2):
                    norm.append(coords[i] / img_width)
                    norm.append(coords[i+1] / img_height)
                out.append(f"{cls_id} " + " ".join(f"{x:.6f}" for x in norm) + "\n")
            except: pass
            
    if not out: return False
    with open(dst_path, "w") as f:
        f.writelines(out)
    return True

# ==========================================
# 3. PGD GENERATOR FUNCTION
# ==========================================
def generate_pgd_image(model, image_path, eps, alpha, iters, device, save_path):
    """Generates and saves a PGD adversarial image without tracking global gradients."""
    device = torch.device(device)
    raw_model = model.model.to(device)
    
    # Enable gradient tracking for the input, but keep weights untouched
    raw_model.train() 

    img = cv2.imread(image_path)
    if img is None: return False
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) 
    
    orig_img_t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    adv_img_t = orig_img_t.clone().detach()

    for _ in range(iters):
        adv_img_t.requires_grad_(True)
        preds = raw_model(adv_img_t)
        
        losses = []
        def collect(x):
            if isinstance(x, torch.Tensor) and x.requires_grad:
                losses.append(x.clone().sum())
            elif isinstance(x, (list, tuple)):
                for j in x: collect(j)
            elif isinstance(x, dict):
                for v in x.values(): collect(v)

        collect(preds)
        if not losses: return False
        
        loss = sum(losses)
        raw_model.zero_grad(set_to_none=True) 
        loss.backward()

        if adv_img_t.grad is not None:
            with torch.no_grad():
                # Take PGD step
                adv_img_t = adv_img_t + alpha * adv_img_t.grad.sign()
                # Project back to epsilon ball
                eta = torch.clamp(adv_img_t - orig_img_t, min=-eps, max=eps)
                # Apply and clamp to valid image range [0, 1]
                adv_img_t = torch.clamp(orig_img_t + eta, min=0, max=1).detach_()

    # Save adversarial image
    adv_img_final = (adv_img_t.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    cv2.imwrite(save_path, adv_img_final) 
    return True

# ==========================================
# 4. MAIN DATASET CONSTRUCTION PIPELINE
# ==========================================
def build_defense_dataset():
    print("--- Starting Defense Dataset Construction ---")
    
    # 1. Create clean workspace
    if os.path.exists(DEFENSE_DIR):
        shutil.rmtree(DEFENSE_DIR)
    os.makedirs(DEFENSE_IMAGES_DIR, exist_ok=True)
    os.makedirs(DEFENSE_LABELS_DIR, exist_ok=True)

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device set to: {DEVICE}")

    # 2. Load Model for Attack Generation
    print("Loading YOLO model for PGD generation...")
    model = YOLO("yolo26n-obb.pt")
    if hasattr(model.model, 'names'): 
        model.model.names = CLASS_NAMES

    # 3. Gather and Sample Images
    all_imgs = [f for f in os.listdir(SOURCE_IMAGES_DIR) if f.endswith(('.png', '.jpg'))]
    all_imgs.sort()
    
    # Only keep images that have a corresponding DOTA label
    valid_images = [f for f in all_imgs if os.path.exists(os.path.join(SOURCE_LABELS_DIR, f.rsplit('.', 1)[0] + '.txt'))]
    
    # Sample the requested amount (e.g., 2000)
    selected_images = random.sample(valid_images, min(len(valid_images), NUM_DEFENSE_IMAGES))
    print(f"Selected {len(selected_images)} source images. Will generate {len(selected_images)*2} total training samples (50% Clean, 50% Adv).")

    # 4. Processing Loop
    success_count = 0
    
    # Using tqdm to show a progress bar
    for idx, filename in enumerate(tqdm(selected_images, desc="Generating Mixed Dataset")):
        base_name = filename.rsplit('.', 1)[0]
        src_img_path = os.path.join(SOURCE_IMAGES_DIR, filename)
        src_lbl_path = os.path.join(SOURCE_LABELS_DIR, base_name + ".txt")
        
        # Output paths for Clean Data
        clean_img_out = os.path.join(DEFENSE_IMAGES_DIR, f"clean_{filename}")
        clean_lbl_out = os.path.join(DEFENSE_LABELS_DIR, f"clean_{base_name}.txt")
        
        # Output paths for Adversarial Data
        adv_img_out = os.path.join(DEFENSE_IMAGES_DIR, f"adv_{filename}")
        adv_lbl_out = os.path.join(DEFENSE_LABELS_DIR, f"adv_{base_name}.txt")

        # --- PROCESS CLEAN IMAGE ---
        img = cv2.imread(src_img_path)
        if img is None: continue
        h, w, _ = img.shape
        
        # Resize clean image and save
        clean_img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(clean_img_out, clean_img_resized)
        
        # Convert and save label for clean image
        valid_label = convert_dota_to_yolo_obb(src_lbl_path, clean_lbl_out, w, h)
        if not valid_label:
            # If label conversion fails (e.g., empty label), delete the image and skip
            os.remove(clean_img_out)
            continue

        # --- PROCESS ADVERSARIAL IMAGE ---
        # Generate PGD attack and save it directly
        attack_success = generate_pgd_image(
            model=model, 
            image_path=src_img_path, 
            eps=DEFENSE_EPSILON, 
            alpha=PGD_ALPHA, 
            iters=PGD_ITERS, 
            device=DEVICE, 
            save_path=adv_img_out
        )
        
        if attack_success:
            # The label is mathematically identical because the attack doesn't move objects.
            # We just copy the clean label file and rename it for the adversarial image.
            shutil.copy(clean_lbl_out, adv_lbl_out)
            success_count += 1
        else:
            # If attack fails, remove the clean files so the dataset stays perfectly balanced
            os.remove(clean_img_out)
            os.remove(clean_lbl_out)

        # --- STRICT MEMORY MANAGEMENT ---
        # Clear VRAM every 50 images to prevent Kaggle T4 OOM crashes
        if idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    # Final cleanuump
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"\nSuccessfully generated {success_count} clean and {success_count} adversarial pairs.")
    print(f"Total images in defense dataset: {success_count * 2}")

    # 5. Create the YAML configuration file for training
    yaml_path = os.path.join(DEFENSE_DIR, "defense_data.yaml")
    yaml_content = {
        "path": os.path.abspath(DEFENSE_DIR),
        "train": "images/train", 
        "val": "images/train",  # We point val to train just to satisfy YOLO structure, but we won't use it.
        "nc": len(CLASS_NAMES),
        "names": list(CLASS_NAMES.values())
    }
    
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_content, f)
        
    print(f"Dataset YAML created at: {yaml_path}")
    print("--- Phase 2 Complete: Ready for Training ---")

if __name__ == "__main__":
    build_defense_dataset()

--- Starting Defense Dataset Construction ---
Device set to: cuda
Loading YOLO model for PGD generation...
Selected 1869 source images. Will generate 3738 total training samples (50% Clean, 50% Adv).


Generating Mixed Dataset: 100%|██████████| 1869/1869 [32:54<00:00,  1.06s/it]


Successfully generated 1868 clean and 1868 adversarial pairs.
Total images in defense dataset: 3736
Dataset YAML created at: /kaggle/working/defense_dataset/defense_data.yaml
--- Phase 2 Complete: Ready for Training ---


## The Defense Trainer

In [ ]:
import os
import torch
import gc
from ultralytics import YOLO

# ==========================================
# PHASE 3: ADVERSARIAL FINE-TUNING (OOM FIX)
# ==========================================
def train_defense_model():
    print("--- Starting Phase 3: Adversarial Fine-Tuning ---")
    
    # 1. Clear VRAM (Extra safety)
    torch.cuda.empty_cache()
    gc.collect()

    # 2. Define Paths
    yaml_path = "/kaggle/working/defense_dataset/defense_data.yaml"
    project_dir = "/kaggle/working/defense_training"
    name = "yolo26n_obb_defended"

    # 3. Load the pre-trained model
    print("Loading base YOLO model...")
    model = YOLO("yolo26n-obb.pt")

    # 4. Execute Multi-GPU Fine-Tuning
    print("Initiating Multi-GPU training on Dual T4s...")
    results = model.train(
        data=yaml_path,
        task='obb',
        epochs=25,               
        imgsz=1024,              
        batch=8,                 # <--- FIX: Lowered from 16 to 8 to prevent OOM
        device=[0, 1],           
        lr0=1e-4,                
        optimizer='auto',        
        project=project_dir,     
        name=name,               
        exist_ok=True,           
        save=True,               
        freeze=10,
        workers=4                # Limits CPU bottlenecks
    )

    print("\n--- Phase 3 Complete: Defense Training Finished ---")
    
    best_weights = os.path.join(project_dir, name, 'weights', 'best.pt')
    print(f"Defended model weights successfully saved to: {best_weights}")

if __name__ == "__main__":
    train_defense_model()

## Evaluating the Defended Model

In [ ]:
import os
import shutil
import random
import yaml
import torch
import cv2
import gc
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ==========================================
# 1. EVALUATION CONFIGURATION
# ==========================================
# Make sure this points to your resumed 'last.pt' or 'best.pt'
DEFENDED_MODEL_PATH = "/kaggle/working/defense_training/yolo26n_obb_defended/weights/best.pt"
REGULAR_MODEL_PATH = "yolo26n-obb.pt"

IMG_SIZE = 1024 
NUM_IMAGES_TO_VALIDATE = 50 
EPSILONS = [0.01, 0.02, 0.04, 0.08]

PGD_ITERS = 10  
PGD_ALPHA_MULTIPLIER = 0.25 

WORK_DIR = "/kaggle/working/ultimate_evaluation"
SOURCE_IMAGES_DIR = "/kaggle/input/dota-1-5/dota/images"
SOURCE_LABELS_DIR = "/kaggle/input/dota-1-5/dota/dota_labels"

CLASS_NAMES = {
    0: 'plane', 1: 'ship', 2: 'storage tank', 3: 'baseball diamond',
    4: 'tennis court', 5: 'basketball court', 6: 'ground track field',
    7: 'harbor', 8: 'bridge', 9: 'large vehicle', 10: 'small vehicle',
    11: 'helicopter', 12: 'roundabout', 13: 'soccer ball field',
    14: 'swimming pool', 15: 'container crane'
}
NAME_TO_IDX = {v: k for k, v in CLASS_NAMES.items()}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def convert_dota_to_yolo_obb(src_path, dst_path, img_width, img_height):
    if not os.path.exists(src_path): return False
    out = []
    with open(src_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 10: continue
            try:
                coords = list(map(float, p[:8]))
                cls = p[8].replace("-", " ")
                if cls not in NAME_TO_IDX: continue
                cls_id = NAME_TO_IDX[cls]
                norm = []
                for i in range(0, 8, 2):
                    norm.append(coords[i] / img_width)
                    norm.append(coords[i+1] / img_height)
                out.append(f"{cls_id} " + " ".join(f"{x:.6f}" for x in norm) + "\n")
            except: pass
    if not out: return False
    with open(dst_path, "w") as f:
        f.writelines(out)
    return True

def generate_targeted_pgd_attack(model, image_path, eps, alpha, iters, device, save_path):
    device = torch.device(device)
    raw_model = model.model.to(device)
    raw_model.train() 

    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) 
    
    orig_img_t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    adv_img_t = orig_img_t.clone().detach()

    for i in range(iters):
        adv_img_t.requires_grad_(True)
        preds = raw_model(adv_img_t)
        
        losses = []
        def collect(x):
            if isinstance(x, torch.Tensor) and x.requires_grad:
                losses.append(x.clone().sum())
            elif isinstance(x, (list, tuple)):
                for j in x: collect(j)
            elif isinstance(x, dict):
                for v in x.values(): collect(v)

        collect(preds)
        if not losses: return False
        
        loss = sum(losses)
        raw_model.zero_grad(set_to_none=True) 
        loss.backward()

        if adv_img_t.grad is not None:
            with torch.no_grad():
                adv_img_t = adv_img_t + alpha * adv_img_t.grad.sign()
                eta = torch.clamp(adv_img_t - orig_img_t, min=-eps, max=eps)
                adv_img_t = torch.clamp(orig_img_t + eta, min=0, max=1).detach_()

    adv_img_final = (adv_img_t.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    cv2.imwrite(save_path, adv_img_final) 
    return True

def calc_f1(precision, recall):
    return 2 * (precision * recall) / (precision + recall + 1e-6)

# ==========================================
# 3. MAIN EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(DEFENDED_MODEL_PATH):
        print(f"ERROR: Could not find defended weights at {DEFENDED_MODEL_PATH}")
        return

    if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on: {DEVICE}")

    # --- LOAD 4 SEPARATE MODELS TO PREVENT FUSION BUGS ---
    print("Loading models...")
    undefended_attack = YOLO(REGULAR_MODEL_PATH)
    undefended_eval   = YOLO(REGULAR_MODEL_PATH)
    defended_attack   = YOLO(DEFENDED_MODEL_PATH)
    defended_eval     = YOLO(DEFENDED_MODEL_PATH)
    
    for m in [undefended_attack, undefended_eval, defended_attack, defended_eval]:
        if hasattr(m.model, 'names'): m.model.names = CLASS_NAMES

    # --- SELECT IMAGES ---
    all_imgs = [f for f in os.listdir(SOURCE_IMAGES_DIR) if f.endswith(('.png', '.jpg'))]
    all_imgs.sort()
    images = [f for f in all_imgs if os.path.exists(os.path.join(SOURCE_LABELS_DIR, f.rsplit('.', 1)[0] + '.txt'))]
    selected_images = random.sample(images, min(len(images), NUM_IMAGES_TO_VALIDATE))
    
    # --- STEP 1: CLEAN BASELINES ---
    print(f"\n--- 1. Establishing Clean Baselines ---")
    temp_clean_dir = os.path.join(WORK_DIR, "clean_data")
    os.makedirs(os.path.join(temp_clean_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(temp_clean_dir, "labels"), exist_ok=True)

    for f in selected_images:
        src_path = os.path.join(SOURCE_IMAGES_DIR, f)
        img = cv2.imread(src_path)
        h, w, _ = img.shape
        cv2.imwrite(os.path.join(temp_clean_dir, "images", f), cv2.resize(img, (IMG_SIZE, IMG_SIZE)))
        convert_dota_to_yolo_obb(
            os.path.join(SOURCE_LABELS_DIR, f.rsplit('.', 1)[0] + ".txt"), 
            os.path.join(temp_clean_dir, "labels", f.rsplit('.', 1)[0] + ".txt"), w, h)

    clean_yaml = os.path.join(WORK_DIR, "clean.yaml")
    with open(clean_yaml, "w") as f:
        yaml.dump({"path": os.path.abspath(temp_clean_dir), "train": "images", "val": "images", "nc": len(CLASS_NAMES), "names": list(CLASS_NAMES.values())}, f)

    print("Evaluating Regular YOLO on Clean Images...")
    clean_reg_metrics = undefended_eval.val(data=clean_yaml, task="obb", device=DEVICE, plots=False, imgsz=IMG_SIZE)
    print("Evaluating Trained YOLO on Clean Images...")
    clean_def_metrics = defended_eval.val(data=clean_yaml, task="obb", device=DEVICE, plots=False, imgsz=IMG_SIZE)

    # --- STEP 2: ADVERSARIAL ATTACKS ---
    results_reg = []
    results_def = []

    for eps in EPSILONS:
        alpha = eps * PGD_ALPHA_MULTIPLIER
        print(f"\n=== Attacking Models (Epsilon: {eps}, Alpha: {alpha}) ===")
        
        # Directories for the two different attacks
        dir_reg_adv = os.path.join(WORK_DIR, f"adv_reg_{eps}")
        dir_def_adv = os.path.join(WORK_DIR, f"adv_def_{eps}")
        for d in [dir_reg_adv, dir_def_adv]:
            os.makedirs(os.path.join(d, "images"), exist_ok=True)
            os.makedirs(os.path.join(d, "labels"), exist_ok=True)
        
        for f in selected_images:
            clean_img_path = os.path.join(temp_clean_dir, "images", f)
            lbl_name = f.rsplit('.', 1)[0] + ".txt"
            
            # Copy labels
            shutil.copy(os.path.join(temp_clean_dir, "labels", lbl_name), os.path.join(dir_reg_adv, "labels", lbl_name))
            shutil.copy(os.path.join(temp_clean_dir, "labels", lbl_name), os.path.join(dir_def_adv, "labels", lbl_name))
            
            # 1. Attack Regular Model
            generate_targeted_pgd_attack(undefended_attack, clean_img_path, eps, alpha, PGD_ITERS, DEVICE, os.path.join(dir_reg_adv, "images", f))
            # 2. Attack Trained Model
            generate_targeted_pgd_attack(defended_attack, clean_img_path, eps, alpha, PGD_ITERS, DEVICE, os.path.join(dir_def_adv, "images", f))
            
        yaml_reg = os.path.join(WORK_DIR, f"adv_reg_{eps}.yaml")
        yaml_def = os.path.join(WORK_DIR, f"adv_def_{eps}.yaml")
        with open(yaml_reg, "w") as f: yaml.dump({"path": os.path.abspath(dir_reg_adv), "train": "images", "val": "images", "nc": len(CLASS_NAMES), "names": list(CLASS_NAMES.values())}, f)
        with open(yaml_def, "w") as f: yaml.dump({"path": os.path.abspath(dir_def_adv), "train": "images", "val": "images", "nc": len(CLASS_NAMES), "names": list(CLASS_NAMES.values())}, f)
            
        # Evaluate Regular Model
        print("Scoring Regular YOLO under attack...")
        m_reg = undefended_eval.val(data=yaml_reg, task="obb", device=DEVICE, plots=False, imgsz=IMG_SIZE)
        results_reg.append({'eps': eps, 'map50': m_reg.box.map50, 'map95': m_reg.box.map, 'p': m_reg.box.mp, 'r': m_reg.box.mr})

        # Evaluate Trained Model
        print("Scoring Trained YOLO under attack...")
        m_def = defended_eval.val(data=yaml_def, task="obb", device=DEVICE, plots=False, imgsz=IMG_SIZE)
        results_def.append({'eps': eps, 'map50': m_def.box.map50, 'map95': m_def.box.map, 'p': m_def.box.mp, 'r': m_def.box.mr})

        # Memory Cleanup
        shutil.rmtree(dir_reg_adv)
        shutil.rmtree(dir_def_adv)
        torch.cuda.empty_cache()
        gc.collect()

    # --- STEP 3: PLOTTING THE COMPARISONS ---
    print("\n--- 3. Generating Side-by-Side Plots ---")
    eps_vals = [r['eps'] for r in results_reg]
    
    # Calculate F1 Scores
    clean_reg_f1 = calc_f1(clean_reg_metrics.box.mp, clean_reg_metrics.box.mr)
    clean_def_f1 = calc_f1(clean_def_metrics.box.mp, clean_def_metrics.box.mr)
    f1_reg_adv = [calc_f1(r['p'], r['r']) for r in results_reg]
    f1_def_adv = [calc_f1(r['p'], r['r']) for r in results_def]

    # Function to create standardized plots
    def make_plot(title, ylabel, clean_reg_val, clean_def_val, adv_reg_vals, adv_def_vals, filename):
        plt.figure(figsize=(10, 6))
        # Baselines (Dashed lines)
        plt.plot(eps_vals, [clean_reg_val]*len(eps_vals), 'r--', label='Clean Regular YOLO', linewidth=2, alpha=0.7)
        plt.plot(eps_vals, [clean_def_val]*len(eps_vals), 'g--', label='Clean Trained YOLO', linewidth=2, alpha=0.7)
        # Attacked values (Solid lines)
        plt.plot(eps_vals, adv_reg_vals, 'r-X', label='Attacked Regular YOLO', linewidth=2, markersize=8)
        
        # ---> FIX APPLIED HERE: Changed 'g-O' to 'g-o' <---
        plt.plot(eps_vals, adv_def_vals, 'g-o', label='Attacked Trained YOLO', linewidth=2, markersize=8)
        
        plt.xlabel('Epsilon (Perturbation Strength)')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(WORK_DIR, filename))
        plt.show()

    # Plot 1: mAP50
    make_plot('mAP50 Comparison: Regular vs. Trained YOLO', 'mAP50', 
              clean_reg_metrics.box.map50, clean_def_metrics.box.map50, 
              [r['map50'] for r in results_reg], [r['map50'] for r in results_def], 'compare_map50.png')

    # Plot 2: mAP95
    make_plot('mAP95 Comparison: Regular vs. Trained YOLO', 'mAP95', 
              clean_reg_metrics.box.map, clean_def_metrics.box.map, 
              [r['map95'] for r in results_reg], [r['map95'] for r in results_def], 'compare_map95.png')

    # Plot 3: Precision (Hallucination Resistance)
    make_plot('Precision Comparison (Hallucination Resistance)', 'Precision', 
              clean_reg_metrics.box.mp, clean_def_metrics.box.mp, 
              [r['p'] for r in results_reg], [r['p'] for r in results_def], 'compare_precision.png')

    # Plot 4: Recall (Cloaking Resistance)
    make_plot('Recall Comparison (Cloaking Resistance)', 'Recall', 
              clean_reg_metrics.box.mr, clean_def_metrics.box.mr, 
              [r['r'] for r in results_reg], [r['r'] for r in results_def], 'compare_recall.png')

    # Plot 5: F1 Score (Overall Accuracy)
    make_plot('F1 Score (Overall Accuracy) Comparison', 'F1 Score', 
              clean_reg_f1, clean_def_f1, 
              f1_reg_adv, f1_def_adv, 'compare_f1.png')

if __name__ == "__main__":
    main()